The survey csv loaded in this notebook has already been anonymised and cleaned.
There were 66 responses for the post-study survey.

In [ ]:
import chardet

# determine encoding using chardet
with open('survey_results_with_application.csv', 'rb') as f:
    result = chardet.detect(f.read())
    print(result)

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import seaborn as sns
import matplotlib.pyplot as plt
import re

survey_analysis_df = pd.read_csv('survey_results_with_application.csv', encoding='utf-8')

survey_analysis_df.head()

In [ ]:
def clean_row(row):
    return row.apply(lambda x: re.sub(r'[^\x00-\x7F]+', ' ', x.strip()) if isinstance(x, str) else x)

# Apply the cleaning function to every row from the second column onwards (excluding the application column)
survey_analysis_df.iloc[:, 1:] = survey_analysis_df.iloc[:, 1:].apply(clean_row, axis=1)
survey_analysis_df.head()

In [ ]:
survey_analysis_df['application'].value_counts()

In [ ]:
# print the column names to check for any issues
print(survey_analysis_df.columns)

In [ ]:
survey_analysis_df.info()

In [ ]:
survey_analysis_df.columns = survey_analysis_df.columns.str.replace(r'[^\x00-\x7F]+', ' ', regex=True)

# Verify the updated column names
print(survey_analysis_df.columns.to_list())

In [ ]:
# print number of columns in survey_analysis_df
print(f"Number of columns in survey_analysis_df: {len(survey_analysis_df.columns)}")

In [ ]:
survey_analysis_df.to_csv('cleaned_survey_analysis_copy.csv', index=False, encoding='utf-8')

In [ ]:
# close ended df is excluding the last 3 columns which are open ended questions
close_ended_df = survey_analysis_df.iloc[:, :-3]
print("Close ended questions columns:", close_ended_df.columns.to_list())
close_ended_df.head()

In [ ]:
import os
# create a directory to save the results
if not os.path.exists('Results/survey_analysis'):
    os.makedirs('Results/survey_analysis')
    print("Directory 'Results/survey_analysis' created.")
else:
    print("Directory 'Results/survey_analysis' already exists.")

In [ ]:
# print number of columns in close ended df excluding application column
print("Number of quantitative questions:", len(close_ended_df.columns) - 1)

In [ ]:
# append question number in front of each SRL col name, e.g Q1. [existing col name]
close_ended_df.columns = [close_ended_df.columns[0]] + [f"Q{i}. {col}" for i, col in enumerate(close_ended_df.columns[1:], start=1)]
print("SRL columns with question numbers:", close_ended_df.columns)

In [ ]:
close_ended_df.info()

In [ ]:
# --- 1. Data Mapping & Preparation ---
# Map the 7-point scale to 3 distinct categories
category_mapping = {
    'Strongly Disagree': 'Disagree',
    'Disagree': 'Disagree',
    'Somewhat Disagree': 'Disagree',
    'Neutral': 'Neutral',
    'Somewhat Agree': 'Agree',
    'Agree': 'Agree',
    'Strongly Agree': 'Agree'
}

# Melt the dataframe (transforms from wide format to long format)
df_melted = close_ended_df.melt(
    id_vars=['application'], 
    value_vars=[c for c in close_ended_df.columns if c != 'application'],
    var_name='Question_Full', 
    value_name='Response'
)

# Explicitly drop the NaN values to execute Pairwise Deletion
df_melted = df_melted.dropna(subset=['Response']).copy()

# Extract just the "Q1", "Q2" part from the full question strings
df_melted['Q_Num'] = df_melted['Question_Full'].apply(lambda x: x.split('.')[0])
df_melted['Category'] = df_melted['Response'].map(category_mapping)

# Calculate counts and convert to percentages 
# The sizes for Q2 will automatically adjust due to the pairwise deletion
counts = df_melted.groupby(['Q_Num', 'application', 'Category']).size().unstack(fill_value=0)
pcts = counts.div(counts.sum(axis=1), axis=0) * 100

# Ensure strict column order for the stacking
for col in ['Disagree', 'Neutral', 'Agree']:
    if col not in pcts.columns:
        pcts[col] = 0
pcts = pcts[['Disagree', 'Neutral', 'Agree']]

# --- 2. Global Styling for Academic Paper ---
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
sns.set_theme(style="white", context="paper", font_scale=1.2)

# Set2 inspired color-blind friendly palette
# Disagree (Set2 Orange), Neutral (Soft Grey), Agree (Set2 Green)
colors = {'Disagree': '#fc8d62', 'Neutral': '#b3b3b3', 'Agree': '#66c2a5'}

srl_phases = {
    'Forethought SRL Phase': ['Q1', 'Q2', 'Q3'],
    'Performance SRL Phase': ['Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9'],
    'Reflection SRL Phase': ['Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15']
}

output_dir = 'Results/survey_analysis'
os.makedirs(output_dir, exist_ok=True)

# --- 3. Plotting Loop for Grouped Stacked Bars ---
for phase_name, questions in srl_phases.items():
    
    # Large enough figure size to give groups room to breathe
    fig, ax = plt.subplots(figsize=(10, 6)) 
    
    x = np.arange(len(questions))  # The label locations
    width = 0.42                   # Thick bars for label visibility
    
    # Sort applications to ensure consistent Left/Right ordering
    applications = sorted(df_melted['application'].unique())
    
    # Iterate through each application to create the grouped side-by-side effect
    for i, app in enumerate(applications):
        # Calculate the offset for side-by-side bars
        offset = x - width/2 if i == 0 else x + width/2
        bottom = np.zeros(len(questions))
        
        for category in ['Disagree', 'Neutral', 'Agree']:
            # Extract values for the current app and category across all questions in this phase
            try:
                values = pcts.loc[(questions, app), category].values
            except KeyError:
                values = np.zeros(len(questions)) # Handle missing groups safely
            
            # Plot the stacked segment
            bars = ax.bar(offset, values, width, bottom=bottom, label=f'{category}' if i==0 else "", 
                          color=colors[category], edgecolor='white', linewidth=0.5)
            
            # Add text annotations inside the bars (only if segment > 5% to avoid clutter)
            for bar, val in zip(bars, values):
                if pd.notna(val) and val > 5:  
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_y() + bar.get_height()/2, 
                            f'{val:.0f}%', ha='center', va='center', color='black', 
                            fontsize=10, fontweight='medium')
            
            bottom += values

    # --- 4. Formatting and Clean Up ---
    ax.set_ylabel('Percentage of Responses (%)', fontweight='bold', labelpad=10)
    ax.set_xlabel('Survey Question', fontweight='bold', labelpad=10)
    ax.set_title(f'Learner Perceptions: {phase_name}', fontweight='bold', pad=20)
    
    ax.set_xticks(x)
    
    # Custom x-tick labels
    xticklabels = [f'{q}' for q in questions]
    ax.set_xticklabels(xticklabels)

    # Set y-axis to exactly 100%
    ax.set_ylim(0, 100)
    
    # Custom Legend
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, title='Response Category', bbox_to_anchor=(1.02, 1), 
              loc='upper left', frameon=False)
    
    # Dynamic annotation about sample size reflecting the pairwise deletion
    app1, app2 = applications[0], applications[1]
    plt.figtext(0.5, -0.05, 
                f"Note: Missing responses excluded item-by-item (Pairwise deletion). Left bar = {app1}, Right bar = {app2}.", 
                ha="center", fontsize=10, style='italic', color='gray')

    sns.despine()
    plt.tight_layout()
    
    # Save as high-resolution PDF and PNG for publications
    filename_base = os.path.join(output_dir, phase_name.replace(' ', '_').lower())
    plt.savefig(f"{filename_base}_stacked.svg", bbox_inches='tight', dpi=300)
    plt.savefig(f"{filename_base}_stacked.png", bbox_inches='tight', dpi=300)
    
    plt.show()

In [ ]:
# include application column and the last 3 columns of survey in open ended df w
open_ended_df = survey_analysis_df[['application']].join(survey_analysis_df.iloc[:, -3:])
open_ended_df.head()

In [ ]:
open_ended_df.info()

In [ ]:
# print how many of each application type there are in each of the 3 questions in the open_ended_df
question_columns = open_ended_df.columns[1:]  # Select the last 3 columns (questions)
for question in question_columns:
    print(f"Counts for question: {question}")
    counts = open_ended_df.groupby(['application'])[question].count()
    print(counts)
    print("\n")